# Exploración de Biodiversidad en GBIF: *Canis lupus* (España y Portugal, 2020–2026)

Este cuaderno consulta la **API pública de ocurrencias de GBIF** (`https://api.gbif.org/v1/occurrence/search`) para obtener registros georreferenciados de *Canis lupus* en España (`ES`) y Portugal (`PT`) entre 2020 y 2026 en rangos altitudinales de 1.000 a 3.000 m, filtra subespecies domésticas (*Canis lupus familiaris*), construye el enlace directo a cada ficha en GBIF (`gbifOccurrenceUrl`) e integra campos Darwin Core avanzados junto con visualización interactiva y exportación a CSV.

In [1]:
import requests
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import time

## 1. Definición de parámetros y columnas Darwin Core

Incluimos `gbifOccurrenceUrl` dentro de `columnas_darwin_core` (construido a partir de `gbifID` con el formato `https://www.gbif.org/occurrence/<gbifID>`) para poder acceder a las fotografías y ficha completa de cada registro y exportarlo directamente en el CSV final.

In [2]:
BASE_URL = "https://api.gbif.org/v1/occurrence/search"

# Parámetros de consulta a la API de GBIF
parametros_base = {
    "taxonKey": 5219173,          # Clave taxonómica oficial de Canis lupus en GBIF
    "country": ["ES", "PT"],
    "year": "2020,2026",
    "hasCoordinate": "true",
    # Quitamos "elevation": "1000,3000" de la API porque descarta registros sin altitud informada
    "limit": 300
}

# Columnas Darwin Core + enlace directo a la ocurrencia en GBIF
columnas_darwin_core = [
    "gbifID",
    "gbifOccurrenceUrl",
    "scientificName",
    "acceptedScientificName",
    "taxonRank",
    "specificEpithet",
    "infraspecificEpithet",
    "countryCode",
    "stateProvince",
    "gadm_level2",
    "decimalLatitude",
    "decimalLongitude",
    "coordinateUncertaintyInMeters",
    "elevation",
    "eventDate",
    "eventTime",
    "year",
    "month",
    "basisOfRecord",
    "individualCount",
    "vitality",
    "hasMedia",
    "mediaType",
    "datasetName",
    "institutionCode",
    "recordedBy",
    "occurrenceID",
    "samplingProtocol",
    "samplingEffort",
    "issues"
]

## 2. Descarga paginada desde la API de GBIF

In [3]:
def descargar_ocurrencias_gbif(params_base, max_registros=3000):
    registros = []
    offset = 0
    limit = params_base.get("limit", 300)

    while True:
        params = params_base.copy()
        params["offset"] = offset
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        resultados = data.get("results", [])
        if not resultados:
            break

        registros.extend(resultados)
        print(f"Descargados {len(registros)} de {data.get('count', '?')} registros...")

        if data.get("endOfRecords", True) or len(registros) >= max_registros:
            break

        offset += limit
        time.sleep(0.2)

    return registros

raw_records = descargar_ocurrencias_gbif(parametros_base)

Descargados 300 de 2686 registros...
Descargados 600 de 2686 registros...
Descargados 900 de 2686 registros...
Descargados 1200 de 2686 registros...
Descargados 1500 de 2686 registros...
Descargados 1800 de 2686 registros...
Descargados 2100 de 2686 registros...
Descargados 2400 de 2686 registros...
Descargados 2686 de 2686 registros...


## 3. Normalización de campos Darwin Core, construcción de `gbifOccurrenceUrl` y filtrado

In [4]:
def normalizar_registro(rec):
    gbif_id = rec.get("gbifID")
    gadm = rec.get("gadm", {}) or {}
    level2 = gadm.get("level2", {}) if isinstance(gadm.get("level2"), dict) else {}
    media = rec.get("media", []) or []
    media_types = sorted({m.get("type") for m in media if isinstance(m, dict) and m.get("type")})

    fila = {
        "gbifID": gbif_id,
        "gbifOccurrenceUrl": f"https://www.gbif.org/occurrence/{gbif_id}" if gbif_id else None,
        "occurrenceID": rec.get("occurrenceID"),
        "scientificName": rec.get("scientificName"),
        "acceptedScientificName": rec.get("acceptedScientificName"),
        "taxonRank": rec.get("taxonRank"),
        "specificEpithet": rec.get("specificEpithet"),
        "infraspecificEpithet": rec.get("infraspecificEpithet"),
        "countryCode": rec.get("countryCode"),
        "stateProvince": rec.get("stateProvince"),
        "gadm_level2": level2.get("name"),
        "decimalLatitude": rec.get("decimalLatitude"),
        "decimalLongitude": rec.get("decimalLongitude"),
        "coordinateUncertaintyInMeters": rec.get("coordinateUncertaintyInMeters"),
        "elevation": rec.get("elevation"),
        "eventDate": rec.get("eventDate"),
        "eventTime": rec.get("eventTime"),
        "year": rec.get("year"),
        "month": rec.get("month"),
        "basisOfRecord": rec.get("basisOfRecord"),
        "individualCount": rec.get("individualCount"),
        "vitality": rec.get("vitality"),
        "samplingProtocol": rec.get("samplingProtocol"),
        "samplingEffort": rec.get("samplingEffort"),
        "datasetName": rec.get("datasetName"),
        "institutionCode": rec.get("institutionCode"),
        "recordedBy": rec.get("recordedBy"),
        "mediaType": ", ".join(media_types) if media_types else None,
        "hasMedia": len(media) > 0,
        "issues": ", ".join(rec.get("issues", []))
    }
    return {col: fila.get(col) for col in columnas_darwin_core}

df_raw = pd.DataFrame([normalizar_registro(r) for r in raw_records])

# Filtrado de perros domésticos (Canis lupus familiaris)
if not df_raw.empty:
    mascara_domestico = (
        df_raw["infraspecificEpithet"].fillna("").str.lower().eq("familiaris")
        | df_raw["scientificName"].fillna("").str.lower().str.contains("familiaris")
    )
    df_lobo = df_raw[~mascara_domestico].copy()
else:
    df_lobo = pd.DataFrame(columns=columnas_darwin_core)

print(f"Total registros filtrados (lobo silvestre): {len(df_lobo)}")
print(f"Registros con multimedia/imágenes disponibles en GBIF: {df_lobo['hasMedia'].sum() if not df_lobo.empty else 0}")
df_lobo[["gbifID", "gbifOccurrenceUrl", "countryCode", "stateProvince", "eventDate", "hasMedia"]].head(10)

Total registros filtrados (lobo silvestre): 1121
Registros con multimedia/imágenes disponibles en GBIF: 801


,gbifID,gbifOccurrenceUrl,countryCode,stateProvince,eventDate,hasMedia
1,6133193494,https://www.gbif.org/occurrence/6133193494,ES,Galicia,2026-01-22T11:38,True
2,6133329298,https://www.gbif.org/occurrence/6133329298,ES,Castilla y León,2026-01-24T11:24:59,True
3,6147664674,https://www.gbif.org/occurrence/6147664674,ES,Galicia,2026-01-31T14:29:08,True
4,6320571023,https://www.gbif.org/occurrence/6320571023,ES,Galicia,2026-01-04T16:21:42,True
6,6159185567,https://www.gbif.org/occurrence/6159185567,ES,Galicia,2026-02-14T14:07:34,True
8,6159222157,https://www.gbif.org/occurrence/6159222157,ES,Galicia,2026-02-14T15:06:27,True
9,6159240975,https://www.gbif.org/occurrence/6159240975,ES,Galicia,2026-02-14T16:31:37,True
11,6159600403,https://www.gbif.org/occurrence/6159600403,ES,Galicia,2026-02-14T14:06:49,True
12,6162771419,https://www.gbif.org/occurrence/6162771419,ES,Galicia,2026-02-22T11:26:16,True
13,6162880185,https://www.gbif.org/occurrence/6162880185,ES,Galicia,2026-02-22T10:51:32,True


## 4. Exportación a CSV con `columnas_darwin_core` (incluyendo `gbifOccurrenceUrl`)

In [5]:
archivo_csv = "canis_lupus_es_pt_2020_2026_darwin_core.csv"
df_lobo[columnas_darwin_core].to_csv(archivo_csv, index=False, encoding="utf-8-sig")
print(f"Dataset exportado correctamente a '{archivo_csv}' con {len(df_lobo.columns)} columnas.")

Dataset exportado correctamente a 'canis_lupus_es_pt_2020_2026_darwin_core.csv' con 30 columnas.


## 5. Mapa interactivo con enlace directo a la ficha y fotos en GBIF

In [6]:
if not df_lobo.empty:
    centro_lat = df_lobo["decimalLatitude"].mean()
    centro_lon = df_lobo["decimalLongitude"].mean()
    mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=6, tiles="OpenStreetMap")
    cluster = MarkerCluster().add_to(mapa)

    for _, row in df_lobo.iterrows():
        url_ficha = row["gbifOccurrenceUrl"]
        etiqueta_media = "📷 Con multimedia" if row["hasMedia"] else "Sin multimedia"
        popup_html = (
            f"<b>{row['scientificName']}</b><br>"
            f"<b>País / Provincia:</b> {row['countryCode']} - {row['stateProvince'] or 'N/D'}<br>"
            f"<b>Fecha:</b> {row['eventDate'] or 'N/D'}<br>"
            f"<b>Altitud:</b> {row['elevation'] or 'N/D'} m<br>"
            f"<b>Multimedia:</b> {etiqueta_media}<br>"
            f"<a href='{url_ficha}' target='_blank'>Ver ficha e imágenes en GBIF</a>"
        )
        folium.Marker(
            location=[row["decimalLatitude"], row["decimalLongitude"]],
            popup=folium.Popup(popup_html, max_width=300),
            icon=folium.Icon(color="green" if row["hasMedia"] else "blue", icon="paw", prefix="fa")
        ).add_to(cluster)

    folium.TileLayer(
        tiles="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
        attr="Map data: &copy; OpenStreetMap contributors, SRTM | Map style: &copy; OpenTopoMap (CC-BY-SA)",
        name="OpenTopoMap (Relieve)",
        overlay=False,
        control=True
    ).add_to(mapa)
    folium.LayerControl().add_to(mapa)        

    mapa.save("mapa_canis_lupus_es_pt.html")
    display(mapa)